# End-to-End Abstractive Text Summarization Benchmarking
### Dataset: SAMSum (Dialogue Summarization)
---
**Project Objective:** This notebook demonstrates an end-to-end machine learning pipeline to fine-tune, evaluate, and benchmark state-of-the-art sequence-to-sequence transformer models (`PEGASUS`, `BART-Large-CNN`, and `Flan-T5-Base`) on conversational text. Performance is quantified using standard abstractive NLP metrics: **ROUGE-1**, **ROUGE-2**, **ROUGE-L**, and **SacreBLEU**.

## 1. Environment Setup & Dependency Installation
---
In this section, we prepare the runtime environment by installing the Hugging Face ecosystem tools (`transformers`, `datasets`, `evaluate`) alongside core acceleration and valuation dependencies.
* We use `nltk` for sentence tokenization during metric scoring.
* `accelerate` handles device placement and optimization automatically.

In [ ]:
!pip install -q transformers[torch] datasets evaluate rouge_score sacrebleu nltk

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 11.5 MB/s eta 0:00:00


In [ ]:
import numpy as np
import torch
import nltk
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer
)
import evaluate

nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True) # Ensures compatibility with your current tokenization setup



True

## 1. Dataset Loading and Exploration
---
We ingest the **SAMSum** dataset, a high-quality corpus containing thousands of messenger-style chats paired with human-written, abstractive summaries. Here, we download the splits (`train`, `validation`, `test`) from the Hugging Face Hub and inspect a few samples.

In [ ]:
from datasets import load_dataset

dataset = load_dataset("knkarthick/samsum")

print(dataset)
# DatasetDict({
#     train:      14732 rows
#     validation: 818 rows
#     test:       819 rows
# })


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/4.36k [00:00<?, ?B/s]

train.csv:   0%|          | 0.00/9.26M [00:00<?, ?B/s]

validation.csv:   0%|          | 0.00/504k [00:00<?, ?B/s]

test.csv:   0%|          | 0.00/522k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/14731 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/818 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/819 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 14731
    })
    validation: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 818
    })
    test: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 819
    })
})


In [ ]:

# look at one example
example = dataset["train"][0]
print("── Dialogue ──")
print(example["dialogue"])
print("\n── Summary ──")
print(example["summary"])

── Dialogue ──
Amanda: I baked  cookies. Do you want some?
Jerry: Sure!
Amanda: I'll bring you tomorrow :-)

── Summary ──
Amanda baked cookies and will bring Jerry some tomorrow.


## 2. Custom Evaluation Metric Framework
---
To accurately score abstractive quality, we instantiate a standalone evaluation pipeline. Because the Hugging Face `Seq2SeqTrainer` masks padding values with `-100` during loss computation, this custom function handles decoding prediction IDs back into raw strings while stripping special tokens and computing **ROUGE** and **SacreBLEU** scores precisely.

In [ ]:
# 2. Setup standard metrics
rouge_metric = evaluate.load("rouge")
bleu_metric = evaluate.load("sacrebleu")
print("Load Metrics Finished")
def compute_metrics(eval_preds, tokenizer):
    preds, labels = eval_preds
    if isinstance(preds, tuple):
        preds = preds[0]

    # Replace negative pad IDs safely
    preds = np.where(preds != -100, preds, tokenizer.pad_token_id)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)

    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    # Format generated sequences sentence by sentence for evaluation
    decoded_preds = ["\n".join(nltk.sent_tokenize(pred.strip())) for pred in decoded_preds]
    decoded_labels = ["\n".join(nltk.sent_tokenize(label.strip())) for label in decoded_labels]

    # Calculate ROUGE scores
    rouge_results = rouge_metric.compute(
        predictions=decoded_preds,
        references=decoded_labels,
        use_stemmer=True
    )

    # Format required structures for SacreBLEU metrics calculation
    bleu_references = [[label] for label in decoded_labels]
    bleu_results = bleu_metric.compute(
        predictions=decoded_preds,
        references=bleu_references
    )

    return {
        "rouge1": round(rouge_results["rouge1"], 4),
        "rouge2": round(rouge_results["rouge2"], 4),
        "rougeL": round(rouge_results["rougeL"], 4),
        "sacrebleu": round(bleu_results["score"], 4)
    }

# Track performance history to determine the ultimate winner
model_leaderboard = {}
best_score = -1
best_model_name = None

# List of the 2 performant models to benchmark against your baseline
candidate_models = [
    "google/pegasus-cnn_dailymail",
    "facebook/bart-large-cnn",
]


Load Metrics Finished


## 3. Iterative Model Training & Benchmarking Loop
---
This is the core training module. We loop through our candidate models (`facebook/bart-large-cnn` and `google/flan-t5-base`), preprocess the dialogue sequences according to their specific tokenization boundaries, and fine-tune them using mixed-precision (`fp16`).

The model optimizing the highest `ROUGE-1` score on the validation set will automatically be saved locally as our champion baseline.

In [ ]:

# 3. Main Loop: Train and evaluate each model dynamically
for model_ckpt in candidate_models:
    print(f"\n{'='*40}\nStarting Fine-Tuning Sequence for: {model_ckpt}\n{'='*40}")

    tokenizer = AutoTokenizer.from_pretrained(model_ckpt)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_ckpt)
    print(f"Load Model Finished : {model_ckpt}")
    # Adjust prefix instructions explicitly for models in the T5 family
    is_t5 = "t5" in model_ckpt.lower()
    prefix = "summarize: " if is_t5 else ""

    # Preprocessing function specific to the model tokenizer context
    def preprocess_function(examples):
        inputs = [prefix + doc for doc in examples["dialogue"]]
        model_inputs = tokenizer(inputs, max_length=512, truncation=True)

        labels = tokenizer(text_target=examples["summary"], max_length=128, truncation=True)
        model_inputs["labels"] = labels["input_ids"]
        return model_inputs

    # Map tokenization rules efficiently across dataset clusters
    tokenized_dataset = dataset.map(preprocess_function, batched=True, remove_columns=dataset["train"].column_names)
    data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

    # Define parameters specific to this run
    clean_name = model_ckpt.replace("/", "-")

    training_args = Seq2SeqTrainingArguments(
        output_dir=f"./results_{clean_name}",
        eval_strategy="epoch",
        save_strategy="epoch",
        learning_rate=5e-5,
        per_device_train_batch_size=4, # Adjust depending on GPU VRAM availability
        per_device_eval_batch_size=4,
        weight_decay=0.01,
        save_total_limit=1,
        num_train_epochs=2,            # Fine-tune efficiently for 3 standard cycles
        predict_with_generate=True,
        fp16=torch.cuda.is_available(), # Activate speed acceleration if using Colab/Kaggle GPU
        logging_steps=100,
        load_best_model_at_end=True,
        metric_for_best_model="rouge1"  # Evaluates best checkpoint via ROUGE-1 precision optimization
    )

    trainer = Seq2SeqTrainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_dataset["train"],
        eval_dataset=tokenized_dataset["validation"],
        processing_class=tokenizer,
        data_collator=data_collator,
        compute_metrics=lambda preds: compute_metrics(preds, tokenizer)
    )

    # Start training
    trainer.train()

    # Evaluate model performance on validation data
    eval_results = trainer.evaluate()
    print(f"Final Validation Scores for {model_ckpt}: {eval_results}")

    # Log scores to leaderboard
    r1_score = eval_results["eval_rouge1"]
    model_leaderboard[model_ckpt] = eval_results

    # 4. Compare and preserve the absolute best model
    if r1_score > best_score:
        best_score = r1_score
        best_model_name = model_ckpt
        print(f"New top performer found! Saving best baseline weights for {model_ckpt}...")
        trainer.save_model("./absolute_best_summarization_model")
        tokenizer.save_pretrained("./absolute_best_summarization_model")

# 5. Output Final Evaluation Leaderboard Results
print(f"\n\n{'='*50}\nFINAL BENCHMARKING LEADERBOARD\n{'='*50}")
for name, metrics in model_leaderboard.items():
    print(f"Model: {name}")
    print(f" -> ROUGE-1: {metrics['eval_rouge1']:.4f} | ROUGE-2: {metrics['eval_rouge2']:.4f} | ROUGE-L: {metrics['eval_rougeL']:.4f} | SacreBLEU: {metrics['eval_sacrebleu']:.4f}\n")

print(f"The winning architecture is: 【 {best_model_name} 】 saved to local directory: './absolute_best_summarization_model'")


Starting Fine-Tuning Sequence for: google/pegasus-cnn_dailymail


config.json:   0%|          | 0.00/1.12k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/88.0 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/1.91M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/65.0 [00:00<?, ?B/s]

...

Map:   0%|          | 0/14731 [00:00<?, ? examples/s]

Map:   0%|          | 0/818 [00:00<?, ? examples/s]

Map:   0%|          | 0/819 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss,Rouge1,Rouge2,Rougel,Sacrebleu
1,1.478003,1.466082,0.405900,0.208400,0.313700,12.295200


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

## 4. Final Model Evaluation on Unseen Test Split
---
To prevent data leakage and establish real-world generalization reliability, we isolate the final saved champion model weights (`./absolute_best_summarization_model`). We verify its true performance by running an evaluation pass over the completely unseen hold-out `test` partition.

In [ ]:

model_path = "./absolute_best_summarization_model"

# tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSeq2SeqLM.from_pretrained(model_path)

# 2. Re-initialize your metrics
# rouge_metric = evaluate.load("rouge")
# bleu_metric = evaluate.load("sacrebleu")


# 3. Prepare the Test Split (accounting for potential T5 prefixing)
is_t5 = "t5" in model.config._name_or_path.lower()
prefix = "summarize: " if is_t5 else ""

def preprocess_function(examples):
    inputs = [prefix + doc for doc in examples["dialogue"]]
    model_inputs = tokenizer(inputs, max_length=512, truncation=True)
    labels = tokenizer(text_target=examples["summary"], max_length=128, truncation=True)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

# Map target configurations explicitly to the 'test' slice
tokenized_test = dataset["test"].map(preprocess_function, batched=True, remove_columns=dataset["test"].column_names)
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

# 4. Setup dummy training arguments tailored specifically for evaluation
eval_args = Seq2SeqTrainingArguments(
    output_dir="./test_evaluation_results",
    per_device_eval_batch_size=8,   # Can be higher than train batch size since it consumes less VRAM
    predict_with_generate=True,
    fp16=torch.cuda.is_available(),
    report_to="none"                # Prevents writing logs to external services like Wandb
)



In [ ]:
# 5. Initialize trainer exclusively pointing to your test set
eval_trainer = Seq2SeqTrainer(
    model=model,
    args=eval_args,
    eval_dataset=tokenized_test,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=lambda preds: compute_metrics(preds)
)
def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple):
        preds = preds[0]

    # Clean up padding tokens
    preds = np.where(preds != -100, preds, tokenizer.pad_token_id)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)

    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    # Sentence tokenize for proper ROUGE tracking
    decoded_preds = ["\n".join(nltk.sent_tokenize(pred.strip())) for pred in decoded_preds]
    decoded_labels = ["\n".join(nltk.sent_tokenize(label.strip())) for label in decoded_labels]

    # Compute scores
    rouge_results = rouge_metric.compute(predictions=decoded_preds, references=decoded_labels, use_stemmer=True)
    bleu_references = [[label] for label in decoded_labels]
    bleu_results = bleu_metric.compute(predictions=decoded_preds, references=bleu_references)

    return {
        "rouge1": round(rouge_results["rouge1"], 4),
        "rouge2": round(rouge_results["rouge2"], 4),
        "rougeL": round(rouge_results["rougeL"], 4),
        "sacrebleu": round(bleu_results["score"], 4)
    }


# 6. Execute Evaluation
print("Running final evaluation pass across the unseen test dataset split...")
test_results = eval_trainer.evaluate()

# 7. Print Out Clean Metrics Table
print(f"\n{'='*45}\n FINAL TEST HOLD-OUT METRICS RESULTS \n{'='*45}")
print(f"  ROUGE-1   Score : {test_results['eval_rouge1']:.4f}")
print(f"  ROUGE-2   Score : {test_results['eval_rouge2']:.4f}")
print(f"  ROUGE-L   Score : {test_results['eval_rougeL']:.4f}")
print(f"  SacreBLEU Score : {test_results['eval_sacrebleu']:.4f}")
print(f"{'='*45}")

## 5. Interactive Inference Sandbox
---
Metrics only tell half the story. In this final section, we load our optimized fine-tuned champion architecture into a sequence generation pipeline to qualitatively test how well it condenses a casual chat sample from the wild.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# 1. Force the correct Sequence-to-Sequence architecture loader
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSeq2SeqLM.from_pretrained(model_path)

# Move to GPU if available
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

# 2. Get your test dialogue
test_sample = dataset["test"][0]["dialogue"]

# 3. Add the prefix ONLY if the winning model happened to be Flan-T5
if "t5" in model.config._name_or_path.lower():
    test_sample = "summarize: " + test_sample

# 4. Tokenize and cleanly generate without conflicting arguments
inputs = tokenizer(test_sample, return_tensors="pt", max_length=512, truncation=True).to(device)

with torch.no_grad():
    summary_ids = model.generate(
        inputs["input_ids"],
        max_new_tokens=60,  # Explicitly using max_new_tokens to avoid the warning
        min_length=15,
        num_beams=4,        # Beam search ensures a clean, cohesive summary sentence
        early_stopping=True
    )

# 5. Decode the output
generated_summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)

print("── Original Dialogue ──")
print(dataset["test"][0]["dialogue"])
print("\n── Best Model Abstractive Summary ──")
print(generated_summary)